<a href="https://colab.research.google.com/github/GabriellaSoratoM/Tcc_2/blob/main/tcc_modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1️⃣ Instalar bibliotecas
# !pip install nltk scikit-learn pandas

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from nltk.stem import RSLPStemmer
import re

# 2️⃣ Stemmer
stemmer = RSLPStemmer()

def preprocessar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúãõâêôç\s]', '', texto)
    palavras = texto.split()
    raizes = [stemmer.stem(p) for p in palavras]
    return " ".join(raizes)

# 3️⃣ Dataset de exemplo
data = {
    "frase": [
        "o adulto é feliz",
        "criança feliz",
        "conteúdo inapropriado",
        "homem trabalhando",
        "frase com palavrão",
        "menino brincando",
        "adulto olhando pornografia",
        "garota estudando",
        "porno",
        "criança bonita",
        "menino no parque",
        "adolescentes discutindo",
        "essa menina é bonitona",
        "beleza pura",
        "bonitão musculoso",
        "video violento",
        "homem sorrindo"
    ],
    "label": [
        "Seguro", "Seguro", "Perigoso", "Seguro", "Perigoso",
        "Seguro", "Perigoso", "Seguro", "Perigoso", "Sensível",
        "Seguro", "Sensível", "Sensível", "Sensível", "Sensível",
        "Perigoso", "Seguro"
    ]
}

df = pd.DataFrame(data)
df["frase_preproc"] = df["frase"].apply(preprocessar_texto)

# 4️⃣ Treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    df["frase_preproc"], df["label"], test_size=0.25, random_state=42
)

# 5️⃣ TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 6️⃣ Logistic Regression
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train_vec, y_train)

# 7️⃣ Avaliar modelo
y_pred = clf.predict(X_test_vec)
print("Relatório de classificação:")
print(classification_report(y_test, y_pred))

# 8️⃣ Função híbrida: regras + aprendizado
# Palavras-chave (stemizadas)
palavras_seguro = ["feliz", "brinc", "estud", "sorr", "trabalh", "parqu"]
palavras_sensiveis = ["sensivel", "adolescent", "crit", "bonit", "belez", "polem"]
palavras_perigoso = ["pornograf", "palavr", "sexual", "inapropri", "violenc", "violent"]

def classificar_frase_hibrida(frase):
    frase_proc = preprocessar_texto(frase)
    raizes = frase_proc.split()

    # 1️⃣ Regras de palavras-chave
    if any(p in raizes for p in palavras_seguro):
        return "Seguro"
    elif any(p in raizes for p in palavras_sensiveis):
        return "Sensível"
    elif any(p in raizes for p in palavras_perigoso):
        return "Perigoso"

    # 2️⃣ Fallback: TF-IDF + Logistic Regression
    frase_vec = vectorizer.transform([frase_proc])
    return clf.predict(frase_vec)[0]

# 9️⃣ Testes
testes = [
    "o adulto está feliz",
    "criança brincando no parque",
    "conteúdo sexual explícito",
    "garota olhando pornografia",
    "criança bonita",
    "video violento",
    "homem sorrindo",
    "adolescentes discutindo",
    "essa menina é bonitona",
    "beleza pura",
    "homem bonito"
]

print("\nTestes de classificação:")
for frase in testes:
    print(f"Frase: '{frase}' -> Classificação: {classificar_frase_hibrida(frase)}")

Relatório de classificação:
              precision    recall  f1-score   support

    Perigoso       0.33      1.00      0.50         1
      Seguro       1.00      0.33      0.50         3
    Sensível       0.00      0.00      0.00         1

    accuracy                           0.40         5
   macro avg       0.44      0.44      0.33         5
weighted avg       0.67      0.40      0.40         5


Testes de classificação:
Frase: 'o adulto está feliz' -> Classificação: Seguro
Frase: 'criança brincando no parque' -> Classificação: Seguro
Frase: 'conteúdo sexual explícito' -> Classificação: Perigoso
Frase: 'garota olhando pornografia' -> Classificação: Perigoso
Frase: 'criança bonita' -> Classificação: Sensível
Frase: 'video violento' -> Classificação: Perigoso
Frase: 'homem sorrindo' -> Classificação: Seguro
Frase: 'adolescentes discutindo' -> Classificação: Perigoso
Frase: 'essa menina é bonitona' -> Classificação: Sensível
Frase: 'beleza pura' -> Classificação: Sensível
Frase: